In [1]:
import numpy as np
import pandas as pd

In [54]:
train_data = pd.read_csv("mnist_train.csv")
test_data = pd.read_csv("mnist_test.csv")

In [55]:
def transform_data(data):
    labels = np.array(data.iloc[:, 0])
    x = np.array(data.iloc[:, 1:], dtype=float)/255

    y = np.zeros((len(data.iloc[:, 0]), 10))
    for i in range(len(labels)):
        temp = np.zeros(10)
        temp[labels[i]] = 1
        y[i] = temp

    return x, y

In [71]:
class MySimpleNN: #Neural Network with 1 hidden layer
    def __init__(self, input_data, labels, hidden_layer_len = 64):
        self.x = input_data
        self.y = labels

        self.w1 = np.random.randn(len(input_data[0]), hidden_layer_len)*0.01
        self.b1 = np.zeros(hidden_layer_len)

        self.w2 = np.random.randn(hidden_layer_len, len(labels[0]))*0.01
        self.b2 = np.zeros(len(labels[0]))

    @staticmethod
    def softmax(z):
        exp_z = np.exp(z - np.max(z))
        return exp_z / np.sum(exp_z)

    @staticmethod
    def adam(lr, grad, first_moment, second_moment, t, beta1 = 0.9, beta2 = 0.999):
        first_moment = beta1 * first_moment + (1 - beta1) * grad
        second_moment = beta2 * second_moment + (1 - beta2) * grad * grad
        first_unbias = first_moment/(1 - beta1 ** t)
        second_unbias = second_moment/(1 - beta2 ** t)
        return (lr * first_unbias / (np.sqrt(second_unbias)+1e-7)), first_moment, second_moment

    def train(self, epochs = 5, lr=0.01):
        X, Y, W1, B1, W2, B2 = self.x, self.y, self.w1, self.b1, self.w2, self.b2
        t = 1
        m_w1, v_w1 = np.zeros_like(self.w1), np.zeros_like(self.w1)
        m_b1, v_b1 = np.zeros_like(self.b1), np.zeros_like(self.b1)
        m_w2, v_w2 = np.zeros_like(self.w2), np.zeros_like(self.w2)
        m_b2, v_b2 = np.zeros_like(self.b2), np.zeros_like(self.b2)

        for epoch in range(epochs):
            epoch_loss = 0.0
            lr = lr * 0.9
            for i in range(len(X)):

                Z1 = X[i] @ W1 + B1
                H = np.maximum(0, Z1)

                Z2 = H @ W2 + B2
                probability = self.softmax(Z2)

                example_loss = -np.sum(self.y[i] * np.log(probability + 1e-9))
                epoch_loss += example_loss

                Dz2 = probability - Y[i]
                Dw2 = np.outer(H, Dz2)
                Db2 = Dz2

                Dz1 = np.dot(W2, Dz2) * (Z1 > 0)
                Dw1 = np.outer(X[i], Dz1)
                Db1 = Dz1

                optimization, m_w2, v_w2 =  self.adam(lr, Dw2, m_w2, v_w2, t)
                W2 -= optimization

                optimization, m_b2, v_b2 =  self.adam(lr, Db2, m_b2, v_b2, t)
                B2 -= optimization

                optimization, m_w1, v_w1 =  self.adam(lr, Dw1, m_w1, v_w1, t)
                W1 -= optimization

                optimization, m_b1, v_b1 =  self.adam(lr, Db1, m_b1, v_b1, t)
                B1 -= optimization

                t += 1
            print(f"Epoch {epoch+1}/{epochs} is completed | Loss: {epoch_loss/len(X)}")

        self.w1, self.b1, self.w2, self.b2 =  W1, B1, W2, B2

    def predict(self, x):
        Z1 = x @ self.w1 + self.b1
        H = np.maximum(0, Z1)

        Z2 = H @ self.w2 + self.b2
        return self.softmax(Z2)

    def evaluate(self, test_x, test_y):
        correct_predictions = 0
        total_samples = len(test_x)

        for i in range(total_samples):
            probabilities = self.predict(test_x[i])
            predicted_class = np.argmax(probabilities)
            true_class = np.argmax(test_y[i])

            if predicted_class == true_class:
                correct_predictions += 1

        accuracy = correct_predictions / total_samples
        return accuracy


In [72]:
x, y = transform_data(train_data)
nn = MySimpleNN(x, y, 128)

In [73]:
nn.train(epochs = 10)

Epoch 1/10 is completed | Loss: 0.5519448370069284
Epoch 2/10 is completed | Loss: 0.4836287656877473
Epoch 3/10 is completed | Loss: 0.4488445552492422
Epoch 4/10 is completed | Loss: 0.421415359257196
Epoch 5/10 is completed | Loss: 0.3920331391992366
Epoch 6/10 is completed | Loss: 0.36801338018939617
Epoch 7/10 is completed | Loss: 0.3486917803355223
Epoch 8/10 is completed | Loss: 0.3299093675930105
Epoch 9/10 is completed | Loss: 0.32013422056561863
Epoch 10/10 is completed | Loss: 0.3005672550820953


In [74]:
test_x, test_y = transform_data(test_data)

print("Predictions for first 10 data entries")
for i in range(10):
    print(f"Prediction: { np.argmax(nn.predict(test_x[i])) } ")
    print(f"Reality: { np.argmax(test_y[i]) }")
print(f"Evaluated Accuracy for all data: {nn.evaluate(test_x, test_y)}")




Predictions for first 10 data entries
Prediction: 2 
Reality: 2
Prediction: 1 
Reality: 1
Prediction: 0 
Reality: 0
Prediction: 4 
Reality: 4
Prediction: 1 
Reality: 1
Prediction: 4 
Reality: 4
Prediction: 9 
Reality: 9
Prediction: 5 
Reality: 5
Prediction: 9 
Reality: 9
Prediction: 0 
Reality: 0
Evaluated Accuracy for all data: 0.9236923692369237
